# Logistic Regression — probabilities, decisions and honest evaluation
LG01–LG06 connect churn prediction to scores, probabilities, log loss, regularization, thresholds and limits. Predict before running the next trusted cell. All rows are fictional, embedded and credential/upload/remote-data free. This notebook is not arbitrary hosted learner execution and never awards web mastery. Genuine Colab remains UNVERIFIED.


## Beginner calculation path — no scientific-library installation required
Run the next cell with Python’s standard library. It applies fixed settings; it does not train a model. Change coefficient to 2, predict the direction first, then explain why score and probability change differently. The synthetic signal has no physical unit.
The full verification section below requires its declared exact reference environment. A version mismatch there is an environment diagnostic, not a failure to understand Logistic Regression.

In [ ]:
import math
signal = math.log(3)
coefficient, baseline = 1, 0
score = coefficient * signal + baseline
probability = 1 / (1 + math.exp(-score))
action = int(probability >= 0.5)
print({"score": score, "probability": probability, "contact": action})
print("Positive-label row loss:", -math.log(probability))


## Verified reference workflow — exact environment
This section is a reproducibility harness, including numerical endpoints and fit receipts; it is not the prerequisite for the introductory calculation. Use a separate environment with Python 3.13.16 and the package profile asserted in the next cell. The matching package command is:
```sh
python -m pip install numpy==2.5.4 pandas==3.0.6 matplotlib==3.11.2 scikit-learn==1.9.1
```
The strict reference assertions remain intentional. Do not change them to claim a passing reference run on another profile. Real Colab execution remains unverified. Development labels may select declared candidates and thresholds; final-test labels must not drive those choices.


In [ ]:
import sys,json,hashlib,warnings,importlib.metadata
import numpy as np
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold,cross_val_score
from sklearn.metrics import log_loss,confusion_matrix
from sklearn.dummy import DummyClassifier
assert sys.version_info[:3]==(3,13,16)
profile={"numpy":"2.5.4","pandas":"3.0.6","matplotlib":"3.11.2","scikit-learn":"1.9.1"}
assert all(importlib.metadata.version(name)==version for name,version in profile.items())
SEED=1729
np.random.seed(SEED)
print(json.dumps({"profile":profile,"seed":SEED},sort_keys=True))


## LG01 — score, probability, action
Predict p and odds for score ln3. Then move only the threshold: should the model refit? For z=w*x+b, increasing the feature x by one adds w to log-odds; increasing w by one at fixed x adds x. Neither is a fixed probability increment. For a different task, class1 could mean documentation Needs Improvement; that is not the churn outcome or validated production labeling.


In [ ]:
score=float(np.log(3))
probability=float(expit(score))
assert np.isclose(probability,.75)
assert np.isclose(probability/(1-probability),3)
assert [int(probability>=threshold) for threshold in [.7,.8]]==[1,0]
assert int(.6>=.6)==1
print(json.dumps({"score":score,"probability":probability,"odds":3,"actions":[1,0]},sort_keys=True))


## LG02 — label-dependent fitting, confidence and endpoints
Calculate loss for y1,p.5 versus p.9. Explain why a final-test label must not drive an update. Keep full arithmetic precision. Explain which data may fit parameters, select declared choices, or evaluate frozen choices. A tiny unregularized row update illustrates a gradient, not sklearn's optimizer.


In [ ]:
positive_loss=float(-np.log(.5))
confident_positive_loss=float(-np.log(.9))
assert np.isclose(positive_loss,np.log(2))
assert confident_positive_loss<positive_loss
training_label=1
gradient=probability-training_label
new_weight=1-.1*gradient*score
new_intercept=-.1*gradient
new_probability=float(expit(new_weight*score+new_intercept))
assert gradient<0 and new_probability>probability
stable_wrong_loss=float(np.logaddexp(0,40))
stored_probability=float(expit(40))
assert stable_wrong_loss==40 and stored_probability==1
clipped_wrong_loss=float(log_loss([0],[stored_probability],labels=[0,1]))
assert np.isfinite(clipped_wrong_loss)
print(json.dumps({"loss_at_half":positive_loss,"gradient":gradient,"new_probability":new_probability,
    "finite_logit_loss":stable_wrong_loss,"exact_stored_p1_y0_loss":{"tag":"infinity","sign":"positive"},
    "library_clipped_loss":clipped_wrong_loss},sort_keys=True))


## Exact fictional partitions
Prediction day0, future outcome window days1–30; positive1=churns, negative0=stays. Features are signed dimensionless usage-change/support-activity indices, not literal negative event counts. Eight training and four development rows match frozen model fixtures; four additional test rows remain outside fitting/selection.


In [ ]:
features_train=np.array([[-2,-1],[-1.5,1],[-1,0],[-0.5,-1],[0.5,1],[1,0],[1.5,-1],[2,1]],dtype=float)
targets_train=np.array([0,0,1,0,1,0,1,1],dtype=int)
features_development=np.array([[-1.2,0.5],[-0.2,-1],[0.8,0.5],[1.8,0]],dtype=float)
targets_development=np.array([0,1,0,1],dtype=int)
features_test=np.array([[-1.8,.3],[-.8,1],[.2,-1],[1.3,.7]],dtype=float)
targets_test=np.array([0,1,0,1],dtype=int)
train_ids=["lg-train1","lg-train2","lg-train3","lg-train4","lg-train5","lg-train6","lg-train7","lg-train8"]
development_ids=["lg-dev1","lg-dev2","lg-dev3","lg-dev4"]
test_ids=["lg-test1","lg-test2","lg-test3","lg-test4"]
bundle={"train":[features_train.tolist(),targets_train.tolist()],
 "development":[features_development.tolist(),targets_development.tolist()],
 "test":[features_test.tolist(),targets_test.tolist()]}
dataset_sha256=hashlib.sha256(json.dumps(bundle,sort_keys=True).encode()).hexdigest()
assert dataset_sha256=="08091e606f3698b473f86f193b76ac8ae4c7047ce25cd2386b05a1e40a06bfde"
assert set(train_ids).isdisjoint(development_ids+test_ids)
print(json.dumps({"rows":16,"data_sha256":dataset_sha256},sort_keys=True))


## LG04 — fit a declared supported workflow
C is inverse L2 strength, not Ridge alpha. The frozen unweighted objective is mean log loss plus ||w||²/(2Cn), intercept unpenalized. Scaling is learned only on the current training set. Unsupported lbfgs/L1 or C0 is not a valid fitted comparison. Nearly separated data/warnings do not certify stable coefficients.


In [ ]:
def pipeline(strength):
    return make_pipeline(StandardScaler(),LogisticRegression(C=strength,
        solver="lbfgs",l1_ratio=0.0,tol=1e-10,max_iter=2000,
        fit_intercept=True,class_weight=None,random_state=SEED,warm_start=False))

def fit_checked(model,features,targets):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model.fit(features,targets)
    assert not caught,[str(item.message) for item in caught]
    assert model.steps[-1][1].n_iter_[0]<2000
    return model

reference=fit_checked(pipeline(1),features_train,targets_train)
assert reference.steps[-1][1].classes_.tolist()==[0,1]
assert np.allclose(reference.steps[-1][1].coef_[0],[0.7405075797076219,0.3245357648341326],atol=1e-9,rtol=1e-9)
assert np.isclose(reference.predict_proba([[.5,1]])[0,1],0.6559137072014022)
print(json.dumps({"anchor_coefficients":reference.steps[-1][1].coef_[0].tolist(),"probability_shape":list(reference.predict_proba(features_development).shape)},sort_keys=True))


## LG05 — fit inside each fold; keep row identity
Predict why a globally learned scaler leaks held-out inputs even without labels. Fit the whole unfitted pipeline separately in each development-training fold. Inspect actual fit IDs rather than assume a pipeline name guarantees validity.


In [ ]:
folds=list(StratifiedKFold(3,shuffle=True,random_state=SEED).split(features_train,targets_train))
fit_receipts=[]
for train_indices,held_indices in folds:
    model=fit_checked(pipeline(1),features_train[train_indices],targets_train[train_indices])
    assert np.allclose(model.steps[0][1].mean_,features_train[train_indices].mean(axis=0))
    fit_ids=[train_ids[index] for index in train_indices]
    held_ids=[train_ids[index] for index in held_indices]
    assert set(fit_ids).isdisjoint(held_ids)
    fit_receipts.append({"scaler_fit_ids":fit_ids,"estimator_fit_ids":fit_ids,"held_ids":held_ids})
assert not np.allclose(np.vstack([features_train,features_development]).mean(axis=0),features_train.mean(axis=0))
print(json.dumps({"fold_receipts":fit_receipts,"global_scaling_valid":False},sort_keys=True))


In [ ]:
strengths=[.1,1,10]
cv_losses=[]
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    for strength in strengths:
        losses=-cross_val_score(pipeline(strength),features_train,targets_train,
            cv=folds,scoring="neg_log_loss",n_jobs=1)
        cv_losses.append(float(losses.mean()))
assert not caught
chosen_strength=strengths[int(np.argmin(cv_losses))]
chosen_model=fit_checked(pipeline(chosen_strength),features_train,targets_train)
class_order=chosen_model.steps[-1][1].classes_.tolist()
positive_column=class_order.index(1)
development_probabilities=chosen_model.predict_proba(features_development)[:,positive_column]
baseline=DummyClassifier(strategy="prior").fit(features_train,targets_train)
baseline_probabilities=baseline.predict_proba(features_development)[:,list(baseline.classes_).index(1)]
print(json.dumps({"candidate_mean_cv_log_loss":cv_losses,"chosen_C":chosen_strength},sort_keys=True))


## LG03 — thresholds, costs and imbalance
Hold probabilities fixed; predict how counts change at equality. Use the explicit p>=threshold action rule, not library predict tie defaults. A prior baseline's p=.5 thus predicts positive at t.5 here. Precision with no predicted positives is undefined, not a silently coerced zero.


In [ ]:
def evaluate(targets,probabilities,threshold):
    actions=(probabilities>=threshold).astype(int)
    true_negative,false_positive,false_negative,true_positive=confusion_matrix(targets,actions,labels=[0,1]).ravel()
    precision=float(true_positive/(true_positive+false_positive)) if true_positive+false_positive else {"tag":"undefined","reason":"no-predicted-positives"}
    recall=float(true_positive/(true_positive+false_negative)) if true_positive+false_negative else {"tag":"undefined","reason":"no-actual-positives"}
    return {"tp":int(true_positive),"fp":int(false_positive),"tn":int(true_negative),"fn":int(false_negative),
        "precision":precision,"recall":recall,"cost":int(false_positive+5*false_negative),
        "accuracy":float(np.mean(actions==targets))}

thresholds=[0,.5,.6,1]
model_actions=[evaluate(targets_development,development_probabilities,threshold) for threshold in thresholds]
baseline_actions=[evaluate(targets_development,baseline_probabilities,threshold) for threshold in thresholds]
chosen_threshold=thresholds[int(np.argmin([item["cost"] for item in model_actions]))]
baseline_threshold=thresholds[int(np.argmin([item["cost"] for item in baseline_actions]))]
assert evaluate(np.array([0,1,1]),np.array([.2,.6,.8]),.6)["fn"]==0
assert isinstance(evaluate(np.array([0,1,1]),np.array([.2,.6,.8]),1)["precision"],dict)
print(json.dumps({"thresholds":thresholds,"model":model_actions,"baseline":baseline_actions,"frozen_threshold":chosen_threshold},sort_keys=True))


### Freeze choices before final evaluation
C was selected by training-fold CV; thresholds use the separate development costs. Keep the eight-row training model fixed, with no refit using development/test labels. Final test outcomes evaluate these frozen workflows once; do not choose another threshold after seeing them. Compare the baseline on exactly the same test rows and cost definition.


In [ ]:
frozen_choice=(chosen_strength,chosen_threshold,baseline_threshold)
final_probabilities=chosen_model.predict_proba(features_test)[:,positive_column]
final_baseline=baseline.predict_proba(features_test)[:,list(baseline.classes_).index(1)]
final_result=evaluate(targets_test,final_probabilities,chosen_threshold)
baseline_result=evaluate(targets_test,final_baseline,baseline_threshold)
assert frozen_choice==(chosen_strength,chosen_threshold,baseline_threshold)
print(json.dumps({"final_test_ids":test_ids,"logistic":final_result,"baseline":baseline_result,
    "final_probability_log_loss":float(log_loss(targets_test,final_probabilities,labels=[0,1]))},sort_keys=True))


## LG06 — probability quality, comparison and limits
Predict whether identical hard labels establish equal probability quality. The following are the frozen authored development probability fixtures, not newly trained algorithms or proof of calibration. Supplied nonlinear candidates can motivate later comparisons, but no KNN/tree implementation is included.


In [ ]:
fixed_targets=np.array([0,0,0,0,0,1,0,1])
confidence_sets=[
    np.array([.1,.2,.3,.4,.5,.6,.8,.9]),
    np.array([.4,.45,.46,.48,.5,.51,.55,.6]),
    np.array([.01,.02,.03,.04,.5,.7,.99,.999])]
accuracy=[float(np.mean((values>=.5)==fixed_targets)) for values in confidence_sets]
probability_losses=[float(log_loss(fixed_targets,values,labels=[0,1])) for values in confidence_sets]
assert accuracy==[.75,.75,.75] and len(set(probability_losses))==3
raw_weights=chosen_model.steps[-1][1].coef_[0]/chosen_model.steps[0][1].scale_
raw_intercept=float(chosen_model.steps[-1][1].intercept_[0]-raw_weights@chosen_model.steps[0][1].mean_)
assert np.allclose(expit(features_test@raw_weights+raw_intercept),final_probabilities,atol=1e-9,rtol=1e-9)
print(json.dumps({"rows":16,"folds":3,"candidates":3,"classes":[0,1],
    "equal_accuracy":accuracy,"different_log_loss":probability_losses,
    "final_cost":final_result["cost"],"data_sha256":dataset_sha256,"mastery":"not awarded by notebook"},sort_keys=True))


## Reduced-scaffolding report
1. Explain score, probability, odds and action in your own words; name the positive class.
2. Contrast finite-logit loss40 with exact stored-probability infinity and numerical clipping.
3. Trace a training label's gradient permission and a fold scaler's fit IDs.
4. Explain why a different threshold does not refit and why test feedback must not select it.
5. Report confusion/costs, baseline protocol, convergence, calibration/shift and small-subgroup uncertainty.
6. Compare a supplied nonlinear candidate fairly without claiming a universally better algorithm or causal intervention.

All data are fictional. No calibrated-risk, fairness, causal or production-performance guarantee. Local success does not verify genuine Colab, screen-reader or browser zoom.
